In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

PREPARED_DIR = Path(
    r"D:\胡溪筼\Documents\2026-7Fall"
    r"\Carolina Data Challenge 2026"
    r"\Graduate_Dataset\prepared"
)

friction_data = pd.read_csv(
    PREPARED_DIR / "time_tax_sdg_matched.csv.gz",
    keep_default_na=False,
    na_values=[""],
    dtype={
        "iso3": "string",
        "country": "string",
        "source": "string",
        "year": "Int64",
        "cross_source_conflict": "boolean",
    },
)

if friction_data.duplicated(["iso3", "year"]).any():
    raise ValueError("Duplicate country-year observations.")

if friction_data["cross_source_conflict"].isna().any():
    raise ValueError("Missing source-conflict flags.")

ACCESS_COLUMNS = {
    "electricity_gap": "wdi__EG.ELC.ACCS.ZS",
    "water_gap": "wdi__SH.H2O.BASW.ZS",
    "cooking_gap": "wdi__EG.CFT.ACCS.ZS",
}

for gap, column in ACCESS_COLUMNS.items():
    values = pd.to_numeric(friction_data[column], errors="raise")

    invalid = values.notna() & ~values.between(0, 100).fillna(False)
    if invalid.any():
        raise ValueError(f"Access values outside 0–100: {column}")

    friction_data[gap] = 1 - values / 100

GAP_COLUMNS = list(ACCESS_COLUMNS)

gap_complete = friction_data[GAP_COLUMNS].notna().all(axis=1)
friction_complete = friction_data.loc[gap_complete].copy()

print(f"Original observations: {len(friction_data)}")
print(f"Complete infrastructure observations: {len(friction_complete)}")
print(f"Entities retained: {friction_complete['iso3'].nunique()}")
print(f"Observations missing at least one component: {(~gap_complete).sum()}")

display(
    friction_complete[
        ["country", "year"] + GAP_COLUMNS
    ].head().round(3)
)

Original observations: 177
Complete infrastructure observations: 168
Entities retained: 86
Observations missing at least one component: 9


,country,year,electricity_gap,water_gap,cooking_gap
0,Albania,2011,0.003,0.082,0.314
1,Argentina,2005,0.033,0.026,0.020
2,Argentina,2010,0.012,0.016,0.008
3,Argentina,2013,0.007,0.011,0.004
4,Armenia,2004,0.009,0.034,0.110


In [2]:
if len(friction_complete) < 4:
    raise ValueError("Too few complete observations for PCA.")

print("Pearson correlations between infrastructure gaps")
display(friction_complete[GAP_COLUMNS].corr().round(3))

gap_values = friction_complete[GAP_COLUMNS].to_numpy(dtype=float)

gap_means = gap_values.mean(axis=0)
gap_scales = gap_values.std(axis=0, ddof=1)

if np.any(gap_scales == 0):
    raise ValueError("An infrastructure component has no variation.")

standardized_gaps = (gap_values - gap_means) / gap_scales

_, singular_values, components = np.linalg.svd(
    standardized_gaps,
    full_matrices=False,
)

explained_variance = singular_values**2 / (len(gap_values) - 1)
explained_ratio = explained_variance / explained_variance.sum()

pc1_weights = components[0].copy()
pc1_scores = standardized_gaps @ pc1_weights

# Orient PC1 so that higher scores align with larger average gaps.
mean_gap = gap_values.mean(axis=1)
alignment = np.corrcoef(pc1_scores, mean_gap)[0, 1]

if alignment < 0:
    pc1_weights *= -1
    pc1_scores *= -1

component_score_correlations = [
    np.corrcoef(standardized_gaps[:, index], pc1_scores)[0, 1]
    for index in range(len(GAP_COLUMNS))
]

pca_diagnostics = pd.DataFrame({
    "component": ["PC1", "PC2", "PC3"],
    "explained_variance_ratio": explained_ratio,
    "cumulative_ratio": np.cumsum(explained_ratio),
})

pca_weights = pd.DataFrame({
    "gap": GAP_COLUMNS,
    "mean_before_standardization": gap_means,
    "sd_before_standardization": gap_scales,
    "pc1_weight": pc1_weights,
    "correlation_with_pc1": component_score_correlations,
})

print("\nExplained variance")
display(pca_diagnostics.round(3))

print("\nPC1 weights and component correlations")
display(pca_weights.round(3))

if not np.all(pc1_weights > 0):
    print(
        "Review required: PC1 does not assign positive weights "
        "to all infrastructure gaps."
    )

Pearson correlations between infrastructure gaps


,electricity_gap,water_gap,cooking_gap
electricity_gap,1.000,0.859,0.815
water_gap,0.859,1.000,0.873
cooking_gap,0.815,0.873,1.000



Explained variance


,component,explained_variance_ratio,cumulative_ratio
0,PC1,0.899,0.899
1,PC2,0.062,0.962
2,PC3,0.038,1.000



PC1 weights and component correlations


,gap,mean_before_standardization,sd_before_standardization,pc1_weight,correlation_with_pc1
0,electricity_gap,0.073,0.191,0.572,0.939
1,water_gap,0.075,0.120,0.585,0.961
2,cooking_gap,0.174,0.289,0.575,0.945


In [3]:
friction_complete["ifi_pca_z"] = (
    pc1_scores - pc1_scores.mean()
) / pc1_scores.std(ddof=1)

friction_complete["ifi_equal_weight"] = (
    friction_complete[GAP_COLUMNS].mean(axis=1)
)

# Keep incomplete observations in the full table with missing index values.
friction_data["ifi_pca_z"] = np.nan
friction_data["ifi_equal_weight"] = np.nan

friction_data.loc[
    friction_complete.index, "ifi_pca_z"
] = friction_complete["ifi_pca_z"]

friction_data.loc[
    friction_complete.index, "ifi_equal_weight"
] = friction_complete["ifi_equal_weight"]

print("Index distributions")
display(
    friction_complete[
        ["ifi_pca_z", "ifi_equal_weight"]
    ].describe().T.round(3)
)

index_comparison = pd.DataFrame([{
    "pearson_r": friction_complete["ifi_pca_z"].corr(
        friction_complete["ifi_equal_weight"]
    ),
    "spearman_rho": (
        friction_complete["ifi_pca_z"].rank()
        .corr(friction_complete["ifi_equal_weight"].rank())
    ),
}])

print("\nAgreement between index constructions")
display(index_comparison.round(3))

detail_columns = [
    "country",
    "year",
    *GAP_COLUMNS,
    "ifi_pca_z",
    "ifi_equal_weight",
    "time_gap_hours",
    "cross_source_conflict",
]

print("\nHighest candidate infrastructure friction")
display(
    friction_complete.nlargest(10, "ifi_pca_z")[detail_columns].round(3)
)

print("\nLowest candidate infrastructure friction")
display(
    friction_complete.nsmallest(10, "ifi_pca_z")[detail_columns].round(3)
)

Index distributions


,count,mean,std,min,25%,50%,75%,max
ifi_pca_z,168.0,0.000,1.00,-0.568,-0.554,-0.460,-0.004,4.054
ifi_equal_weight,168.0,0.108,0.19,0.000,0.002,0.019,0.109,0.824



Agreement between index constructions


,pearson_r,spearman_rho
0,0.997,0.997



Highest candidate infrastructure friction


,country,year,electricity_gap,water_gap,cooking_gap,ifi_pca_z,ifi_equal_weight,time_gap_hours,cross_source_conflict
153,Tanzania,2006,0.851,0.631,0.991,4.054,0.824,3.000,False
52,Ethiopia,2013,0.692,0.611,0.966,3.675,0.756,3.050,False
117,Malawi,2005,0.934,0.419,0.978,3.561,0.777,1.783,False
154,Tanzania,2014,0.765,0.509,0.972,3.512,0.749,2.933,False
87,Cambodia,2004,0.858,0.417,0.941,3.372,0.739,2.700,False
96,Liberia,2010,0.947,0.308,0.996,3.279,0.750,0.892,False
97,Lesotho,2003,0.946,0.339,0.795,3.125,0.693,2.250,False
155,Uganda,2018,0.581,0.470,0.988,3.083,0.680,1.700,False
92,Lao PDR,2003,0.520,0.451,0.983,2.909,0.651,1.800,False
93,Lao PDR,2008,0.340,0.353,0.971,2.277,0.555,2.100,False



Lowest candidate infrastructure friction


,country,year,electricity_gap,water_gap,cooking_gap,ifi_pca_z,ifi_equal_weight,time_gap_hours,cross_source_conflict
10,Belgium,2005,0.0,0.0,0.0,-0.568,0.0,1.383,False
11,Belgium,2013,0.0,0.0,0.0,-0.568,0.0,1.400,False
24,Switzerland,2000,0.0,0.0,0.0,-0.568,0.0,1.987,False
25,Switzerland,2004,0.0,0.0,0.0,-0.568,0.0,1.866,False
28,Switzerland,2013,0.0,0.0,0.0,-0.568,0.0,1.476,False
41,Germany,2013,0.0,0.0,0.0,-0.568,0.0,1.440,False
42,Denmark,2001,0.0,0.0,0.0,-0.568,0.0,1.067,False
53,Finland,2000,0.0,0.0,0.0,-0.568,0.0,1.392,False
54,Finland,2010,0.0,0.0,0.0,-0.568,0.0,1.131,False
58,United Kingdom,2015,0.0,0.0,0.0,-0.568,0.0,1.365,False


In [4]:
friction_analysis = friction_data.copy()

RENAME_COLUMNS = {
    "gender__SL.TLF.CACT.FE.ZS": "female_lfpr_pct",
    "gender__SL.TLF.CACT.MA.ZS": "male_lfpr_pct",
    "wdi__NY.GDP.PCAP.PP.KD": "gdp_per_capita_ppp",
    "wdi__SP.DYN.TFRT.IN": "fertility_rate",
    "wdi__SP.URB.TOTL.IN.ZS": "urban_population_pct",
}

friction_analysis = friction_analysis.rename(columns=RENAME_COLUMNS)

friction_analysis["lfpr_gap_pp"] = (
    friction_analysis["male_lfpr_pct"]
    - friction_analysis["female_lfpr_pct"]
)

gdp = friction_analysis["gdp_per_capita_ppp"]

if (gdp.notna() & gdp.le(0)).any():
    raise ValueError("GDP must be positive before log transformation.")

friction_analysis["log_gdp_per_capita"] = np.log(gdp)

friction_views = {
    "All available observations": friction_analysis,
    "Exclude source conflicts": friction_analysis.loc[
        ~friction_analysis["cross_source_conflict"]
    ].copy(),
    "Latest observation per entity": (
        friction_analysis
        .sort_values(["iso3", "year"])
        .drop_duplicates("iso3", keep="last")
        .copy()
    ),
}

view_summary = []

for name, frame in friction_views.items():
    available = frame["ifi_pca_z"].notna()

    view_summary.append({
        "view": name,
        "time_observations": len(frame),
        "observations_with_ifi": int(available.sum()),
        "entities_with_ifi": frame.loc[
            available, "iso3"
        ].nunique(),
    })

display(pd.DataFrame(view_summary))

,view,time_observations,observations_with_ifi,entities_with_ifi
0,All available observations,177,168,86
1,Exclude source conflicts,172,163,86
2,Latest observation per entity,89,86,86


In [5]:
IFI_COLUMNS = ["ifi_pca_z", "ifi_equal_weight"]

IFI_TARGETS = [
    "female_unpaid_hours",
    "male_unpaid_hours",
    "time_gap_hours",
    "female_lfpr_pct",
    "lfpr_gap_pp",
]

IFI_CONTROLS = [
    "log_gdp_per_capita",
    "year",
    "fertility_rate",
    "urban_population_pct",
]


def pearson_correlation(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    if len(x) < 3 or np.std(x) == 0 or np.std(y) == 0:
        return np.nan

    return float(np.corrcoef(x, y)[0, 1])


def partial_correlation(frame, x_column, y_column, controls):
    control_values = frame[controls].to_numpy(dtype=float)
    scales = control_values.std(axis=0)

    if np.any(scales == 0):
        raise ValueError("A control variable has no variation.")

    standardized = (
        control_values - control_values.mean(axis=0)
    ) / scales

    design = np.column_stack([
        np.ones(len(frame)),
        standardized,
    ])

    if np.linalg.matrix_rank(design) < design.shape[1]:
        raise ValueError("Control matrix is not full rank.")

    if len(frame) <= design.shape[1] + 1:
        raise ValueError("Too few observations for partial correlation.")

    residuals = []

    for column in [x_column, y_column]:
        values = frame[column].to_numpy(dtype=float)

        residual = values - design @ np.linalg.lstsq(
            design, values, rcond=None
        )[0]

        original_norm = np.linalg.norm(values - values.mean())

        if (
            original_norm == 0
            or np.linalg.norm(residual) <= 1e-10 * original_norm
        ):
            return np.nan

        residuals.append(residual)

    return pearson_correlation(*residuals)


ifi_association_rows = []

for view_name, frame in friction_views.items():
    for target in IFI_TARGETS:
        # Both index constructions use exactly the same records.
        required = IFI_COLUMNS + [target] + IFI_CONTROLS

        numeric = (
            frame[required]
            .apply(pd.to_numeric, errors="raise")
            .replace([np.inf, -np.inf], np.nan)
        )

        complete = numeric.notna().all(axis=1)

        analysis = frame.loc[
            complete, ["iso3", "country"]
        ].join(numeric.loc[complete])

        for index_column in IFI_COLUMNS:
            x = analysis[index_column]
            y = analysis[target]

            ifi_association_rows.append({
                "view": view_name,
                "index": index_column,
                "related_variable": target,
                "observations": len(analysis),
                "entities": analysis["iso3"].nunique(),
                "pearson_r": pearson_correlation(x, y),
                "spearman_rho": pearson_correlation(
                    x.rank(method="average"),
                    y.rank(method="average"),
                ),
                "partial_pearson_r": partial_correlation(
                    analysis,
                    index_column,
                    target,
                    IFI_CONTROLS,
                ),
            })

ifi_associations = pd.DataFrame(ifi_association_rows)

for view_name in friction_views:
    print(f"\nIFI associations: {view_name}")

    with pd.option_context(
        "display.max_columns", None,
        "display.max_rows", None,
    ):
        display(
            ifi_associations.loc[
                ifi_associations["view"].eq(view_name)
            ]
            .drop(columns="view")
            .reset_index(drop=True)
            .round(3)
        )


IFI associations: All available observations


,index,related_variable,observations,entities,pearson_r,spearman_rho,partial_pearson_r
0,ifi_pca_z,female_unpaid_hours,166,85,-0.187,0.169,-0.423
1,ifi_equal_weight,female_unpaid_hours,166,85,-0.201,0.156,-0.450
2,ifi_pca_z,male_unpaid_hours,166,85,-0.359,-0.464,-0.017
3,ifi_equal_weight,male_unpaid_hours,166,85,-0.369,-0.469,-0.034
4,ifi_pca_z,time_gap_hours,166,85,0.084,0.482,-0.449
5,ifi_equal_weight,time_gap_hours,166,85,0.078,0.479,-0.465
6,ifi_pca_z,female_lfpr_pct,166,85,0.369,-0.009,0.531
7,ifi_equal_weight,female_lfpr_pct,166,85,0.366,-0.006,0.520
8,ifi_pca_z,lfpr_gap_pp,166,85,-0.168,0.080,-0.562
9,ifi_equal_weight,lfpr_gap_pp,166,85,-0.167,0.083,-0.549



IFI associations: Exclude source conflicts


,index,related_variable,observations,entities,pearson_r,spearman_rho,partial_pearson_r
0,ifi_pca_z,female_unpaid_hours,161,85,-0.209,0.178,-0.449
1,ifi_equal_weight,female_unpaid_hours,161,85,-0.223,0.165,-0.479
2,ifi_pca_z,male_unpaid_hours,161,85,-0.364,-0.458,-0.020
3,ifi_equal_weight,male_unpaid_hours,161,85,-0.373,-0.463,-0.038
4,ifi_pca_z,time_gap_hours,161,85,0.073,0.489,-0.470
5,ifi_equal_weight,time_gap_hours,161,85,0.066,0.486,-0.488
6,ifi_pca_z,female_lfpr_pct,161,85,0.376,-0.016,0.534
7,ifi_equal_weight,female_lfpr_pct,161,85,0.373,-0.012,0.522
8,ifi_pca_z,lfpr_gap_pp,161,85,-0.172,0.072,-0.562
9,ifi_equal_weight,lfpr_gap_pp,161,85,-0.171,0.075,-0.549



IFI associations: Latest observation per entity


,index,related_variable,observations,entities,pearson_r,spearman_rho,partial_pearson_r
0,ifi_pca_z,female_unpaid_hours,85,85,-0.275,0.068,-0.429
1,ifi_equal_weight,female_unpaid_hours,85,85,-0.295,0.036,-0.469
2,ifi_pca_z,male_unpaid_hours,85,85,-0.381,-0.496,0.130
3,ifi_equal_weight,male_unpaid_hours,85,85,-0.388,-0.506,0.115
4,ifi_pca_z,time_gap_hours,85,85,0.017,0.380,-0.497
5,ifi_equal_weight,time_gap_hours,85,85,0.003,0.364,-0.526
6,ifi_pca_z,female_lfpr_pct,85,85,0.449,0.112,0.661
7,ifi_equal_weight,female_lfpr_pct,85,85,0.449,0.112,0.654
8,ifi_pca_z,lfpr_gap_pp,85,85,-0.167,0.105,-0.591
9,ifi_equal_weight,lfpr_gap_pp,85,85,-0.166,0.112,-0.581


In [6]:
INTERACTION_CONTROLS = [
    "log_gdp_per_capita",
    "year",
    "fertility_rate",
    "urban_population_pct",
]

INDEX_COLUMNS = ["ifi_pca_z", "ifi_equal_weight"]

required = (
    ["time_gap_hours", "female_lfpr_pct"]
    + INDEX_COLUMNS
    + INTERACTION_CONTROLS
)

reference = (
    friction_analysis[required]
    .apply(pd.to_numeric, errors="raise")
    .replace([np.inf, -np.inf], np.nan)
    .dropna()
)

# Fixed reference values for all subsequent comparisons.
time_center = reference["time_gap_hours"].mean()

index_centers = reference[INDEX_COLUMNS].mean()
index_scales = reference[INDEX_COLUMNS].std(ddof=1)

if index_scales.le(0).any():
    raise ValueError("An index has no variation.")

control_centers = reference[INTERACTION_CONTROLS].mean()
control_scales = reference[INTERACTION_CONTROLS].std(ddof=1)

if control_scales.le(0).any():
    raise ValueError("A background variable has no variation.")


def fit_interaction(frame, index_column):
    time_centered = (
        frame["time_gap_hours"].to_numpy(dtype=float) - time_center
    )
    index_standardized = (
        frame[index_column].to_numpy(dtype=float)
        - index_centers[index_column]
    ) / index_scales[index_column]

    controls = (
        (
            frame[INTERACTION_CONTROLS] - control_centers
        ) / control_scales
    ).to_numpy(dtype=float)

    design = np.column_stack([
        np.ones(len(frame)),
        time_centered,
        index_standardized,
        time_centered * index_standardized,
        controls,
    ])

    if len(frame) <= design.shape[1]:
        raise ValueError("Too few observations for the interaction model.")

    if np.linalg.matrix_rank(design) < design.shape[1]:
        raise ValueError("Interaction design matrix is not full rank.")

    outcome = frame["female_lfpr_pct"].to_numpy(dtype=float)

    coefficients = np.linalg.lstsq(
        design, outcome, rcond=None
    )[0]

    return coefficients


interaction_rows = []
interaction_fits = {}
interaction_data = {}

for view_name, frame in friction_views.items():
    numeric = (
        frame[required]
        .apply(pd.to_numeric, errors="raise")
        .replace([np.inf, -np.inf], np.nan)
    )

    complete = numeric.notna().all(axis=1)

    analysis = frame.loc[
        complete, ["iso3", "country"]
    ].join(numeric.loc[complete])

    interaction_data[view_name] = analysis

    for index_column in INDEX_COLUMNS:
        coefficients = fit_interaction(analysis, index_column)

        interaction_fits[(view_name, index_column)] = coefficients

        interaction_rows.append({
            "view": view_name,
            "index": index_column,
            "observations": len(analysis),
            "entities": analysis["iso3"].nunique(),
            "time_slope_at_reference_mean_ifi": coefficients[1],
            "interaction_coefficient": coefficients[3],
        })

interaction_summary = pd.DataFrame(interaction_rows)

with pd.option_context("display.max_columns", None):
    display(interaction_summary.round(3))

,view,index,observations,entities,time_slope_at_reference_mean_ifi,interaction_coefficient
0,All available observations,ifi_pca_z,166,85,-6.032,1.805
1,All available observations,ifi_equal_weight,166,85,-6.092,1.956
2,Exclude source conflicts,ifi_pca_z,161,85,-6.280,1.902
3,Exclude source conflicts,ifi_equal_weight,161,85,-6.363,2.050
4,Latest observation per entity,ifi_pca_z,85,85,-5.893,1.040
5,Latest observation per entity,ifi_equal_weight,85,85,-5.862,1.275


In [7]:
IFI_LEVELS = {
    "P25": 0.25,
    "P50": 0.50,
    "P90": 0.90,
}

slope_rows = []

for (view_name, index_column), coefficients in interaction_fits.items():
    analysis = interaction_data[view_name]

    for level_name, quantile in IFI_LEVELS.items():
        index_value = reference[index_column].quantile(quantile)

        index_z = (
            index_value - index_centers[index_column]
        ) / index_scales[index_column]

        # Slope of female LFPR with respect to Time Tax.
        slope = coefficients[1] + coefficients[3] * index_z

        # Descriptive support check around the evaluation point.
        observed_z = (
            analysis[index_column] - index_centers[index_column]
        ) / index_scales[index_column]

        nearby = analysis.loc[
            (observed_z - index_z).abs().le(0.5)
        ]

        slope_rows.append({
            "view": view_name,
            "index": index_column,
            "ifi_level": level_name,
            "ifi_original_scale": index_value,
            "ifi_reference_z": index_z,
            "time_slope_pp_per_hour": slope,
            "nearby_observations": len(nearby),
            "nearby_entities": nearby["iso3"].nunique(),
            "nearby_time_gap_min": nearby["time_gap_hours"].min(),
            "nearby_time_gap_max": nearby["time_gap_hours"].max(),
        })

interaction_slopes = pd.DataFrame(slope_rows)

for view_name in friction_views:
    print(f"\nTime Tax slopes by IFI level: {view_name}")

    with pd.option_context("display.max_columns", None):
        display(
            interaction_slopes.loc[
                interaction_slopes["view"].eq(view_name)
            ]
            .drop(columns="view")
            .reset_index(drop=True)
            .round(3)
        )


Time Tax slopes by IFI level: All available observations


,index,ifi_level,ifi_original_scale,ifi_reference_z,time_slope_pp_per_hour,nearby_observations,nearby_entities,nearby_time_gap_min,nearby_time_gap_max
0,ifi_pca_z,P25,-0.555,-0.553,-7.029,122,59,0.502,5.462
1,ifi_pca_z,P50,-0.465,-0.463,-6.867,127,63,0.502,5.462
2,ifi_pca_z,P90,1.111,1.104,-4.039,10,8,0.850,4.333
3,ifi_equal_weight,P25,0.001,-0.557,-7.181,124,63,0.502,5.462
4,ifi_equal_weight,P50,0.019,-0.465,-7.001,126,64,0.502,5.462
5,ifi_equal_weight,P90,0.358,1.311,-3.528,10,8,0.850,4.333



Time Tax slopes by IFI level: Exclude source conflicts


,index,ifi_level,ifi_original_scale,ifi_reference_z,time_slope_pp_per_hour,nearby_observations,nearby_entities,nearby_time_gap_min,nearby_time_gap_max
0,ifi_pca_z,P25,-0.555,-0.553,-7.331,118,59,0.714,5.462
1,ifi_pca_z,P50,-0.465,-0.463,-7.160,123,63,0.714,5.462
2,ifi_pca_z,P90,1.111,1.104,-4.180,10,8,0.850,4.333
3,ifi_equal_weight,P25,0.001,-0.557,-7.503,120,63,0.714,5.462
4,ifi_equal_weight,P50,0.019,-0.465,-7.315,122,64,0.714,5.462
5,ifi_equal_weight,P90,0.358,1.311,-3.676,10,8,0.850,4.333



Time Tax slopes by IFI level: Latest observation per entity


,index,ifi_level,ifi_original_scale,ifi_reference_z,time_slope_pp_per_hour,nearby_observations,nearby_entities,nearby_time_gap_min,nearby_time_gap_max
0,ifi_pca_z,P25,-0.555,-0.553,-6.467,59,59,0.502,4.783
1,ifi_pca_z,P50,-0.465,-0.463,-6.374,63,63,0.502,4.783
2,ifi_pca_z,P90,1.111,1.104,-4.744,4,4,0.850,4.333
3,ifi_equal_weight,P25,0.001,-0.557,-6.572,63,63,0.502,4.783
4,ifi_equal_weight,P50,0.019,-0.465,-6.455,64,64,0.502,4.783
5,ifi_equal_weight,P90,0.358,1.311,-4.191,4,4,0.850,4.333


In [8]:
interaction_loo_rows = []

for view_name, analysis in interaction_data.items():
    for index_column in INDEX_COLUMNS:
        baseline = interaction_fits[(view_name, index_column)]

        low_value = reference[index_column].quantile(0.25)
        high_value = reference[index_column].quantile(0.90)

        low_z = (
            low_value - index_centers[index_column]
        ) / index_scales[index_column]

        high_z = (
            high_value - index_centers[index_column]
        ) / index_scales[index_column]

        for iso3 in sorted(analysis["iso3"].unique()):
            remaining = analysis.loc[analysis["iso3"].ne(iso3)]

            country = analysis.loc[
                analysis["iso3"].eq(iso3), "country"
            ].iloc[0]

            try:
                coefficients = fit_interaction(remaining, index_column)

                beta_interaction = coefficients[3]
                low_slope = coefficients[1] + beta_interaction * low_z
                high_slope = coefficients[1] + beta_interaction * high_z
                diagnostic = ""

            except ValueError as error:
                beta_interaction = np.nan
                low_slope = np.nan
                high_slope = np.nan
                diagnostic = str(error)

            interaction_loo_rows.append({
                "view": view_name,
                "index": index_column,
                "excluded_country": country,
                "removed_observations": len(analysis) - len(remaining),
                "baseline_interaction": baseline[3],
                "interaction_after_exclusion": beta_interaction,
                "low_ifi_slope": low_slope,
                "high_ifi_slope": high_slope,
                "slope_difference_high_minus_low": high_slope - low_slope,
                "absolute_interaction_change":
                    abs(beta_interaction - baseline[3]),
                "diagnostic": diagnostic,
            })

interaction_loo_details = pd.DataFrame(interaction_loo_rows)
interaction_loo_summary_rows = []

for (view_name, index_column), group in interaction_loo_details.groupby(
    ["view", "index"], sort=False
):
    valid = group.dropna(subset=["interaction_after_exclusion"])

    if valid.empty:
        raise ValueError(f"No valid exclusions: {view_name}, {index_column}")

    largest_change = valid.loc[
        valid["absolute_interaction_change"].idxmax()
    ]

    interaction_loo_summary_rows.append({
        "view": view_name,
        "index": index_column,
        "baseline_interaction": valid["baseline_interaction"].iloc[0],
        "minimum_interaction": valid["interaction_after_exclusion"].min(),
        "maximum_interaction": valid["interaction_after_exclusion"].max(),
        "positive_interaction_exclusions": int(
            valid["interaction_after_exclusion"].gt(0).sum()
        ),
        "valid_exclusions": len(valid),
        "failed_exclusions": len(group) - len(valid),
        "minimum_high_minus_low_slope":
            valid["slope_difference_high_minus_low"].min(),
        "maximum_high_minus_low_slope":
            valid["slope_difference_high_minus_low"].max(),
        "most_influential_exclusion":
            largest_change["excluded_country"],
    })

interaction_loo_summary = pd.DataFrame(interaction_loo_summary_rows)

with pd.option_context("display.max_columns", None):
    display(interaction_loo_summary.round(3))

,view,index,baseline_interaction,minimum_interaction,maximum_interaction,positive_interaction_exclusions,valid_exclusions,failed_exclusions,minimum_high_minus_low_slope,maximum_high_minus_low_slope,most_influential_exclusion
0,All available observations,ifi_pca_z,1.805,0.864,2.547,85,85,0,1.432,4.220,Tanzania
1,All available observations,ifi_equal_weight,1.956,0.972,2.897,85,85,0,1.815,5.408,Lao PDR
2,Exclude source conflicts,ifi_pca_z,1.902,0.956,2.640,85,85,0,1.584,4.374,Tanzania
3,Exclude source conflicts,ifi_equal_weight,2.050,1.075,2.984,85,85,0,2.007,5.571,Lao PDR
4,Latest observation per entity,ifi_pca_z,1.040,0.572,1.843,85,85,0,0.947,3.054,Pakistan
5,Latest observation per entity,ifi_equal_weight,1.275,0.641,2.362,85,85,0,1.196,4.410,Pakistan


In [9]:
latest_for_pca = (
    friction_data
    .sort_values(["iso3", "year"])
    .drop_duplicates("iso3", keep="last")
    .dropna(subset=GAP_COLUMNS)
    .copy()
)

latest_gaps = latest_for_pca[GAP_COLUMNS].to_numpy(dtype=float)

latest_means = latest_gaps.mean(axis=0)
latest_scales = latest_gaps.std(axis=0, ddof=1)

if len(latest_gaps) < 4 or np.any(latest_scales == 0):
    raise ValueError("Insufficient variation for the alternative PCA.")

latest_standardized = (latest_gaps - latest_means) / latest_scales

_, latest_singular_values, latest_components = np.linalg.svd(
    latest_standardized, full_matrices=False
)

latest_weights = latest_components[0].copy()
latest_pc1 = latest_standardized @ latest_weights

if np.corrcoef(latest_pc1, latest_gaps.mean(axis=1))[0, 1] < 0:
    latest_weights *= -1

latest_explained_ratio = (
    latest_singular_values**2 / (latest_singular_values**2).sum()
)

all_complete_gaps = friction_complete[GAP_COLUMNS].to_numpy(dtype=float)

alternative_scores = (
    (all_complete_gaps - latest_means) / latest_scales
) @ latest_weights

alternative_index = friction_complete[["iso3", "year"]].copy()
alternative_index["ifi_latest_pca"] = alternative_scores

print("Alternative PCA")
print(f"Entities used: {len(latest_for_pca)}")
print(f"PC1 explained variance: {latest_explained_ratio[0]:.3f}")

display(pd.DataFrame({
    "gap": GAP_COLUMNS,
    "original_pc1_weight": pc1_weights,
    "latest_entity_pc1_weight": latest_weights,
}).round(3))

if not np.all(latest_weights > 0):
    print("Review required: alternative PC1 contains nonpositive weights.")

original_scores = friction_complete["ifi_pca_z"].to_numpy(dtype=float)

print("Agreement with original PCA")
display(pd.DataFrame([{
    "pearson_r": np.corrcoef(original_scores, alternative_scores)[0, 1],
    "spearman_rho": pd.Series(original_scores).rank().corr(
        pd.Series(alternative_scores).rank()
    ),
}]))

# Calibrate the alternative index on the same full analysis reference.
alternative_reference = interaction_data[
    "All available observations"
].merge(
    alternative_index,
    on=["iso3", "year"],
    how="left",
    validate="one_to_one",
)

if alternative_reference["ifi_latest_pca"].isna().any():
    raise ValueError("Alternative index missing for reference observations.")

alternative_center = alternative_reference["ifi_latest_pca"].mean()
alternative_scale = alternative_reference["ifi_latest_pca"].std(ddof=1)

alternative_low = alternative_reference["ifi_latest_pca"].quantile(0.25)
alternative_high = alternative_reference["ifi_latest_pca"].quantile(0.90)

alternative_low_z = (alternative_low - alternative_center) / alternative_scale
alternative_high_z = (alternative_high - alternative_center) / alternative_scale

pca_sensitivity_rows = []

for view_name, analysis in interaction_data.items():
    alternative = analysis.merge(
        alternative_index,
        on=["iso3", "year"],
        how="left",
        validate="one_to_one",
    )

    if alternative["ifi_latest_pca"].isna().any():
        raise ValueError(f"Alternative index missing: {view_name}")

    alternative_z = (
        alternative["ifi_latest_pca"] - alternative_center
    ) / alternative_scale

    # Reuse the existing fitter with its fixed reference scaling.
    alternative["ifi_pca_z"] = (
        alternative_z * index_scales["ifi_pca_z"]
        + index_centers["ifi_pca_z"]
    )

    coefficients = fit_interaction(alternative, "ifi_pca_z")

    pca_sensitivity_rows.append({
        "view": view_name,
        "observations": len(alternative),
        "original_interaction":
            interaction_fits[(view_name, "ifi_pca_z")][3],
        "alternative_interaction": coefficients[3],
        "alternative_low_ifi_slope":
            coefficients[1] + coefficients[3] * alternative_low_z,
        "alternative_high_ifi_slope":
            coefficients[1] + coefficients[3] * alternative_high_z,
    })

pca_sensitivity = pd.DataFrame(pca_sensitivity_rows)

with pd.option_context("display.max_columns", None):
    display(pca_sensitivity.round(3))

Alternative PCA
Entities used: 86
PC1 explained variance: 0.903


,gap,original_pc1_weight,latest_entity_pc1_weight
0,electricity_gap,0.572,0.578
1,water_gap,0.585,0.580
2,cooking_gap,0.575,0.574


Agreement with original PCA


,pearson_r,spearman_rho
0,0.999902,0.999919


,view,observations,original_interaction,alternative_interaction,alternative_low_ifi_slope,alternative_high_ifi_slope
0,All available observations,166,1.805,1.790,-7.075,-4.047
1,Exclude source conflicts,161,1.902,1.889,-7.382,-4.187
2,Latest observation per entity,85,1.040,0.994,-6.496,-4.815


In [10]:
RESULTS_DIR = (
    PREPARED_DIR.parent
    / "analysis_results"
    / "infrastructure_friction"
)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

result_tables = {
    "pca_diagnostics": pca_diagnostics,
    "pca_weights": pca_weights,
    "ifi_associations": ifi_associations,
    "interaction_summary": interaction_summary,
    "interaction_slopes": interaction_slopes,
    "interaction_leave_one_entity_out": interaction_loo_details,
    "interaction_sensitivity_summary": interaction_loo_summary,
    "alternative_pca_sensitivity": pca_sensitivity,
}

for name, table in result_tables.items():
    table.to_csv(RESULTS_DIR / f"{name}.csv", index=False)

friction_analysis.to_csv(
    RESULTS_DIR / "infrastructure_friction_analysis.csv.gz",
    index=False,
    compression="gzip",
)

print(f"Saved {len(result_tables)} result tables and the analysis data.")
print(f"Output directory: {RESULTS_DIR}")

Saved 8 result tables and the analysis data.
Output directory: D:\胡溪筼\Documents\2026-7Fall\Carolina Data Challenge 2026\Graduate_Dataset\analysis_results\infrastructure_friction
